# rgzn · Kaggle 环境验收（A 岗）

本地装不了 torch，`check_pipeline.py` 的第 8 项（Dataset / 增强 / DataLoader）一直是 SKIP。
这个 notebook 在 Kaggle 上把它补验到 8/8。

**用法**：New Notebook → 右侧 Add Data 挂载 `abdallahalidev/plantvillage-dataset` → 逐格运行。

GPU 不必开，CPU 跑完只要几十秒。

In [ ]:
import os
import subprocess

REPO = "/kaggle/working/rgzn"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/a-ying-w/rgzn.git", REPO], check=True)

os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print(os.getcwd())

In [ ]:
from pathlib import Path

# 在 /kaggle/input 下自动找含 color/ 的目录，避免数据集 slug 变化
candidates = [p.parent for p in Path("/kaggle/input").rglob("color") if p.is_dir()]
assert candidates, "没找到 color/ 目录：先在右侧 Add Data 挂载 plantvillage 数据集"

DATA_ROOT = str(candidates[0])
print("DATA_ROOT =", DATA_ROOT)
print("classes:", sorted(p.name for p in (Path(DATA_ROOT) / "color").iterdir() if p.is_dir())[:6])
print("segmented 存在:", (Path(DATA_ROOT) / "segmented").is_dir())

In [ ]:
# 8 项全绿（这次第 8 项不再 SKIP）
import subprocess

proc = subprocess.run(
    ["python", "scripts/check_pipeline.py", "--root", DATA_ROOT],
    capture_output=True,
    text=True,
)
print(proc.stdout)
print(proc.stderr)
assert proc.returncode == 0, "有检查项 FAIL，看上面的输出"
assert "SKIP" not in proc.stdout, "第 8 项被跳过了，说明 torch 不可用"

In [ ]:
# 确认增强配方与类别加权所需的接口都在
import torch
from src.config import load_config
from src.data.dataset import build_loaders
from src.data.transforms import AUG_RECIPES

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("aug recipes:", AUG_RECIPES)

cfg = load_config("configs/e0_resnet18_ft.yaml")
cfg["data"]["root"] = DATA_ROOT
cfg["data"]["num_workers"] = 2
cfg["train"]["batch_size"] = 32

loaders = build_loaders(cfg, splits=("val",))
x, y = next(iter(loaders["val"]))
print("batch:", tuple(x.shape), "labels:", y[:8].tolist())
assert tuple(x.shape[1:]) == (3, 224, 224)
print("A 岗接口就绪：8/8 通过")

In [ ]:
# 正式跑 E0：先开 GPU（Settings -> Accelerator -> GPU T4 x2）和 Internet
import pathlib
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.train",
    "--config", "configs/e0_resnet18_ft.yaml",
    "--seed", "42",
    "--set", f"data.root={DATA_ROOT}",
]
try:
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=5400)
    print(proc.stdout[-4000:])
    print(proc.stderr[-2000:])
    assert proc.returncode == 0, "训练失败，看上面的报错"
finally:
    rows = pathlib.Path("results/results.csv").read_text().strip().splitlines()
    print("results.csv 最后 3 行：")
    print("\n".join(rows[-3:]))

## 跑完做两件事

1. 截图最后 3 行（含 val / test 两行数字）发群里，这是全组第一条基线。
2. 把 `results/results.csv` 新增的两行提交到 GitHub（`runs/` 已 gitignore，不用管）。

如果显存不够报 `CUDA out of memory`，先 `--set train.batch_size=32` 再跑一次。